# MARL cephalometric landmarks: training on Kaggle

**Before running** (Settings panel on the right):
1. **Accelerator: GPU T4 x2** (or P100).
2. **Internet: On** (needed to download Aariz and pip packages).
3. **Add data** (two Kaggle datasets you upload once, see the chat instructions):
   - `ceph-marl-code`: the `ceph-marl.zip` project.
   - `isbi2015-raw`: a zip of your `raw/ISBI2015` folder (RawImage + 400_junior + 400_senior).

Run the cells top to bottom. Cells 1-4 take ~15 min the first time; cell 5 is the training check.

In [ ]:
# 1. Find the code, copy it to the writable working folder, install the one missing package
import glob, os, shutil, subprocess, sys
src = os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/scripts/train.py', recursive=True)[0]))
dst = '/kaggle/working/ceph-marl'
if not os.path.exists(dst):
    shutil.copytree(src, dst)
os.chdir(dst)
!pip install -q segmentation-models-pytorch
import torch
print('code:', src, '->', dst)
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 2. ISBI 2015: locate the uploaded raw folder (Kaggle unzips uploads automatically) and prepare it
isbi_raw = os.path.dirname(glob.glob('/kaggle/input/**/400_junior', recursive=True)[0])
print('ISBI raw folder:', isbi_raw)
if not os.path.exists('data/isbi/train/meta.json'):
    !python scripts/prepare_data.py isbi --raw "{isbi_raw}" --out data/isbi

In [ ]:
# 3. Aariz: download from the official figshare record (1.95 GB) and prepare it.
#    If the download fails, upload Aariz.zip as a third Kaggle dataset instead; this cell then finds it.
found = glob.glob('/kaggle/input/**/cephalogram_machine_mappings.csv', recursive=True)
if found:
    aariz_raw = os.path.dirname(found[0])
else:
    if not os.path.exists('/kaggle/tmp/Aariz.zip'):
        os.makedirs('/kaggle/tmp', exist_ok=True)
        !wget -q --show-progress -O /kaggle/tmp/Aariz.zip "https://ndownloader.figshare.com/files/51041642"
        !cd /kaggle/tmp && unzip -q -o Aariz.zip
    aariz_raw = os.path.dirname(glob.glob('/kaggle/tmp/**/cephalogram_machine_mappings.csv', recursive=True)[0])
print('Aariz raw folder:', aariz_raw)
if not os.path.exists('data/aariz/train/meta.json'):
    !python scripts/prepare_data.py aariz --raw "{aariz_raw}" --out data/aariz

In [ ]:
# 4. Quick correctness check (about 1 minute)
!python -m pytest -q tests

## 5. Training check: one short ISBI DQN run
Validation error should drop below **7.9 mm** (the mean-shape baseline from the data analysis).
The log also tells us the speed (agent-steps per second), which sets the budget for the full plan.

In [ ]:
!python scripts/train.py --config configs/isbi_dqn.yaml --out runs/check_isbi_dqn --set agent_steps=300000 eval_every=50000
!python scripts/evaluate.py --checkpoint runs/check_isbi_dqn/best.pt --split val

In [ ]:
# Learning curve + speed
import json, matplotlib.pyplot as plt
log = [json.loads(l) for l in open('runs/check_isbi_dqn/log.jsonl')]
steps = [r['step'] for r in log]; mre = [r['val_MRE_mm'] for r in log]
plt.plot(steps, mre, 'o-'); plt.axhline(7.88, ls='--', c='gray', label='mean-shape start (7.9 mm)')
plt.xlabel('agent-steps'); plt.ylabel('validation MRE (mm)'); plt.legend(); plt.show()
print(f"speed: {steps[-1] / log[-1]['time_s']:.0f} agent-steps/s -> 3M-step run = {3e6 / (steps[-1] / log[-1]['time_s']) / 3600:.1f} h")

## 6. Full experiment plan (run in separate sessions)
Kaggle sessions stop after 12 h, so run one block per session and download `runs/` after each
(cell 7). Set `agent_steps` from the speed printed above. Examples:

In [ ]:
# Example: one method, one seed. Change config / seed per session.
# !python scripts/train.py --config configs/isbi_mappo.yaml --seed 0
# !python scripts/train.py --config configs/isbi_unet.yaml --seed 0
# Evaluate a finished run (val + test splits + cross-dataset):
# run = 'runs/isbi_mappo_mappo_s0'
# for split in ['val', 'test1', 'test2']:
#     !python scripts/evaluate.py --checkpoint {run}/best.pt --split {split}
# !python scripts/evaluate.py --checkpoint {run}/best.pt --data-root data/aariz --split test

In [ ]:
# 7. Pack results for download (Output tab -> runs_results.zip). Excludes nothing important: checkpoints are small.
shutil.make_archive('/kaggle/working/runs_results', 'zip', 'runs')
print('download /kaggle/working/runs_results.zip from the Output tab')